# Example: alternate-optima degeneracy in FBA

Demonstrates the phenomenon behind `07_fluxes.ipynb`'s "many fluxes read 0 when they shouldn't"
issue: `model.optimize()` only fixes the objective (growth rate). When a model has redundant
capacity (isozymes, alternate transporters/routes), many different flux distributions hit that
same growth rate, and the LP solver just returns whichever vertex it reaches first - not
necessarily a unique or biologically meaningful one.

Below, the same LP (same model, bounds, objective) is solved with three different Gurobi
algorithms - primal simplex, dual simplex, barrier - standing in for "rerunning `optimize()`":
Gurobi's real default (`Method=-1`, automatic/concurrent) can itself pick different algorithms
run to run depending on thread scheduling. All three hit the *identical* objective value but
disagree, sometimes sharply, on individual fluxes. `pFBA` then collapses all three into one
answer.

In [1]:
import os
import pandas as pd
import numpy as np
from cobra.io import read_sbml_model
from cobra.flux_analysis import pfba

final_dir = os.path.join("..", "data", "models", "02_final_annotated")
MODEL_NAME = "ecBSU1"  # heavily isozyme-duplicated (GECKO-style), makes degeneracy easy to see

medium = pd.read_csv("../data/datasets/M9_media.csv")
GLUCOSE_UPTAKE = 7.63  # matches 07_fluxes.ipynb's Chubukov et al. (2013) condition
medium.loc[medium["ID"] == "EX_glc__D_e", "LOWER_BOUND"] = -GLUCOSE_UPTAKE


def load_model():
    model = read_sbml_model(os.path.join(final_dir, f"{MODEL_NAME}.xml"))
    for _, row in medium.iterrows():
        if row["ID"] in model.reactions:
            model.reactions.get_by_id(row["ID"]).bounds = (row["LOWER_BOUND"], row["UPPER_BOUND"])
    return model


baseline = load_model()
baseline_growth = baseline.slim_optimize()
print(f"[INFO] {MODEL_NAME} baseline optimal growth: {baseline_growth:.6f} 1/h")

'' is not a valid SBML 'SId'.


Set parameter Username


Set parameter LicenseID to value 2695449


--------------------------------------------


--------------------------------------------


Academic license - for non-commercial use only - expires 2026-08-13


[INFO] ecBSU1 baseline optimal growth: 0.567985 1/h


## Solve the same LP with three different Gurobi algorithms

`Params.Method`: `0` = primal simplex, `1` = dual simplex, `2` = barrier. Each is a legitimate way
to solve the exact same optimization problem - none is "more correct" than another - so if they
disagree on individual fluxes, that disagreement can only come from degeneracy in the problem
itself, not from a bug.

In [2]:
METHODS = {0: "Primal simplex", 1: "Dual simplex", 2: "Barrier"}

method_solutions = {}
method_objectives = {}
for method_id, method_name in METHODS.items():
    model = load_model()
    model.solver.problem.Params.Method = method_id
    solution = model.optimize()
    method_solutions[method_name] = solution.fluxes
    method_objectives[method_name] = solution.objective_value
    print(f"[OK] {method_name}: objective = {solution.objective_value:.6f}")

print()
print("All three methods reach the same objective value:",
      len(set(round(v, 6) for v in method_objectives.values())) == 1)

'' is not a valid SBML 'SId'.


[OK] Primal simplex: objective = 0.567985


'' is not a valid SBML 'SId'.


[OK] Dual simplex: objective = 0.567985


'' is not a valid SBML 'SId'.


[OK] Barrier: objective = 0.567985

All three methods reach the same objective value: True


## Compare individual reaction fluxes across the three solves

Same model, same bounds, same objective value - but very different internal flux distributions.
We look at a handful of central-carbon reactions (and their `_reverse` siblings, where this
GECKO-style model splits reversible reactions into two irreversible halves).

In [3]:
WATCH_REACTIONS = [
    "PGM", "PGM_reverse",
    "FBAf", "FBAf_reverse",
    "TPI_num1", "TPI_reverse_num1", "TPI_num2", "TPI_reverse_num2",
    "PYK",
    "SUCOAS", "SUCOAS_reverse",
    "FUM", "FUM_reverse",
    "CS_num1", "CS_num2",
]

comparison = pd.DataFrame({
    method_name: [fluxes.get(rid, np.nan) for rid in WATCH_REACTIONS]
    for method_name, fluxes in method_solutions.items()
}, index=WATCH_REACTIONS)
comparison["range (max-min)"] = comparison[METHODS.values()].max(axis=1) - comparison[METHODS.values()].min(axis=1)
comparison = comparison.round(3)
comparison

,Primal simplex,Dual simplex,Barrier,range (max-min)
PGM,0.000,0.000,992.058,992.058
PGM_reverse,7.942,7.942,1000.000,992.058
FBAf,0.000,0.000,1000.000,1000.000
FBAf_reverse,0.000,0.000,992.444,992.444
TPI_num1,0.000,0.000,993.293,993.293
TPI_reverse_num1,0.000,6.707,1000.000,1000.000
TPI_num2,0.000,0.000,1000.000,1000.000
TPI_reverse_num2,6.707,0.000,1000.000,1000.000
PYK,0.000,0.000,0.000,0.000
SUCOAS,0.000,988.732,0.000,988.732


A large `range` marks a degenerate reaction: the model is genuinely indifferent between it and a
parallel alternative, so different (equally valid) solver algorithms make different choices. A
flux reading `0.0` in one column and nonzero in another is exactly the "reads 0 but probably
isn't" symptom seen in `07_fluxes.ipynb`.

`Barrier` is the extreme case: several fluxes sit at ~`1000` (the bound). That's a **futile
cycle** - plain FBA never forbids a loop that consumes/regenerates the same metabolites net-zero
unless a loopless constraint is added. Barrier's solution routes a huge, self-cancelling loop
through `PGM`/`TPI`/`FBAf`/`FUM_reverse` on top of the real pathway flux - biomass is exactly as
optimal with or without it, so nothing stops it.

## pFBA collapses the degeneracy

`pfba()` adds a second-stage objective - minimize total flux subject to the already-found optimal
growth rate - which picks out a single, parsimonious point instead of an arbitrary vertex. Running
it after each of the three solves above gives (essentially) the same flux distribution regardless
of which algorithm found the initial optimum.

In [4]:
pfba_solutions = {}
for method_id, method_name in METHODS.items():
    model = load_model()
    model.solver.problem.Params.Method = method_id
    solution = pfba(model)
    pfba_solutions[method_name] = solution.fluxes
    print(f"[OK] pFBA after {method_name}: objective = {model.slim_optimize():.6f}, "
          f"total flux = {solution.fluxes.abs().sum():.4f}")

pfba_comparison = pd.DataFrame({
    method_name: [fluxes.get(rid, np.nan) for rid in WATCH_REACTIONS]
    for method_name, fluxes in pfba_solutions.items()
}, index=WATCH_REACTIONS)
pfba_comparison["range (max-min)"] = (
    pfba_comparison[METHODS.values()].max(axis=1) - pfba_comparison[METHODS.values()].min(axis=1)
)
pfba_comparison.round(3)

'' is not a valid SBML 'SId'.


[OK] pFBA after Primal simplex: objective = 0.567985, total flux = 625.2364


'' is not a valid SBML 'SId'.


[OK] pFBA after Dual simplex: objective = 0.567985, total flux = 625.2364


'' is not a valid SBML 'SId'.


[OK] pFBA after Barrier: objective = 0.567985, total flux = 625.2361


,Primal simplex,Dual simplex,Barrier,range (max-min)
PGM,0.000,0.000,0.000,0.000
PGM_reverse,7.942,7.942,7.942,0.000
FBAf,7.057,7.057,7.057,0.000
FBAf_reverse,0.000,0.000,0.000,0.000
TPI_num1,0.000,0.000,0.000,0.000
TPI_reverse_num1,0.000,6.707,6.707,6.707
TPI_num2,0.000,0.000,0.000,0.000
TPI_reverse_num2,6.707,0.000,0.000,6.707
PYK,0.000,0.000,0.000,0.000
SUCOAS,0.000,0.000,0.000,0.000


`range` is now ~0 for almost every reaction - pFBA's minimal-total-flux tiebreak lands on the
same solution regardless of which algorithm found the initial optimum, and the `Barrier` futile
cycle is gone entirely (a loop only ever adds to total flux). This is why `07_fluxes.ipynb` uses
`pfba()` instead of `model.optimize()` for per-reaction flux reporting.

`TPI_num1`/`TPI_num2` (and `_reverse` siblings) are the one pair still disagreeing under pFBA -
literal isozyme duplicates, so routing flux through either costs the same in the "minimize total
flux" sum. pFBA removes *which pathway* is used but can't break a tie between two copies of the
*same* reaction - the same degeneracy, one level down.

---

# Addendum: chasing down the unreal growth rates in `iBsu1103`, `iBsu1103v2`, `iBsu1209`

`03_simulations.ipynb` reported implausible growth for three models:

| model | reported growth (all conditions) |
|---|---|
| `iBsu1103` | ~407 1/h, constant regardless of substrate |
| `iBsu1103v2` | ~227 1/h, constant regardless of substrate |
| `iBsu1209` | 0 1/h, always |

This is exploratory debugging only - nothing here is fed back into `03_simulations.ipynb` or the
`02_final_annotated` / `00_initial` files on disk. Everything below builds its own throwaway
`cobra.Model` copy in memory.

## `iBsu1103` / `iBsu1103v2`: a hidden third exchange layer

`set_culture_media()` (in `03_simulations.ipynb`) only edits the ~14 `EX_*_e` reactions listed in
the "M9 media" sheet and leaves every other exchange at whatever the model file shipped with.
For most models (`iYO844`, `iBsu1147`, ...) that is fine, because their `EX_*_e` reactions **are**
the real system boundary and default to closed (`lb=0`) for anything not explicitly fed.

`iBsu1103`/`iBsu1103v2` are different. Inspecting `model.exchanges` shows they carry **three**
tiers of compartments instead of two - cytosol (`_c`) to extracellular (`_e`) to an unregistered,
uncounted `_b` "boundary" pool (not even listed in `model.compartments`) - and the *true* system
boundary is the `EX_<id>_b` reaction (`<met>_b <=>`, a genuine single-metabolite exchange), not
`EX_<id>_e` (which is just an internal `_e <=> _b` relay, itself left wide open by default:
244/500 such relays sit at `(-10000, 10000)`). Since `set_culture_media()` never touches the `_b`
layer, ~244 metabolites stay freely importable no matter what medium is nominally applied - hence
the huge, condition-independent growth.

Two more one-off bugs compound this once the `_b` layer is closed down:
- `EX_k_e` collides with a second, unrelated reaction of the same auto-generated ID (`k_e <=> k_c`,
  an internal transporter) - the *real* potassium boundary reaction got shoved to `EX_k_e_2`.
- `R_H2Otg` (`h2o_c --> h2o_e`) is irreversible **export-only**, so water can never be imported
  even when `EX_h2o_e`/`EX_h2o_b` are open. `iBsu1103v2` also has `EX_nh4_e` as `nh4_e --> nh4_b`
  (irreversible the wrong way), blocking ammonium import.

None of this is visible from `model.exchanges` bounds alone - it only shows up by tracing each
M9 metabolite from `_e` down to its actual `_b` terminus and checking reaction directionality.

In [5]:
import os
import pandas as pd
from cobra.io import read_sbml_model

final_dir = os.path.join("..", "data", "models", "02_final_annotated")
medium_df = pd.read_csv("../data/datasets/M9_media.csv")
media_ids = set(medium_df["ID"])


def load_fixed_1103(name):
    # Apply M9 at the real `_b` boundary layer instead of the misleading `_e` relay.
    model = read_sbml_model(os.path.join(final_dir, f"{name}.xml"))

    # 1. close every true terminal boundary reaction (single metabolite, compartment '_b')
    terminal = [r for r in model.reactions
                if len(r.metabolites) == 1 and next(iter(r.metabolites)).id.endswith("_b")]
    for r in terminal:
        r.bounds = (0, 1000)

    # 2. open only the M9 compounds, at their *_b* boundary reaction
    for _, row in medium_df.iterrows():
        base = row["ID"].replace("EX_", "").replace("_e", "")
        bid = f"EX_{base}_b"
        if bid in model.reactions:
            model.reactions.get_by_id(bid).bounds = (row["LOWER_BOUND"], row["UPPER_BOUND"])

    # 3. one-off directionality fixes found by tracing blocked biomass precursors
    if "R_H2Otg" in model.reactions:
        model.reactions.get_by_id("R_H2Otg").bounds = (-10000, 10000)
    ex_nh4 = model.reactions.get_by_id("EX_nh4_e") if "EX_nh4_e" in model.reactions else None
    if ex_nh4 is not None and ex_nh4.lower_bound >= 0:
        ex_nh4.bounds = (-10000, 10000)

    return model


for name in ["iBsu1103", "iBsu1103v2"]:
    model = load_fixed_1103(name)
    print(f"{name}: growth on glucose M9 = {model.slim_optimize():.4f} 1/h "
          f"(was ~{407 if name == 'iBsu1103' else 227} 1/h before the fix)")

iBsu1103: growth on glucose M9 = 0.6214 1/h (was ~407 1/h before the fix)


iBsu1103v2: growth on glucose M9 = 0.6254 1/h (was ~227 1/h before the fix)


Both now land in the same realistic range as `iYO844`/`iBsu1147` (~0.6 1/h on glucose M9,
vs. real *B. subtilis* doubling times of roughly 1-2 h). The next cell reruns the actual
Chubukov et al. (2013) condition table through the fixed models to confirm growth now
*responds* to the carbon source instead of sitting at a constant, substrate-independent value.

In [6]:
chubukov = pd.read_csv("../data/datasets/Chubukov_2013.csv")


def run_experiment_b_layer(model, experiment_data):
    growth = []
    for _, row in experiment_data.iterrows():
        reactions = [r.strip() for r in str(row["reactions"]).split(";")]
        fluxes = [float(f.strip()) for f in str(row["import fluxes"]).split(";")]
        with model:
            model.reactions.get_by_id("EX_glc__D_b").lower_bound = 0.0
            for rxn_id, flux in zip(reactions, fluxes):
                base = rxn_id.replace("EX_", "").replace("_e", "")
                bid = f"EX_{base}_b"
                if bid in model.reactions:
                    model.reactions.get_by_id(bid).lower_bound = flux
                else:
                    print(f"  [WARN] {model.id}: no boundary reaction for {rxn_id}")
            growth.append(model.optimize().objective_value)
    return growth


for name in ["iBsu1103", "iBsu1103v2"]:
    model = load_fixed_1103(name)
    result = chubukov[["condition", "experimental growth"]].copy()
    result[name] = run_experiment_b_layer(model, chubukov)
    print(result.round(3), "\n")

  [WARN] mergem_iBsu1121VCorrected_trans_bigg: no boundary reaction for EX_glcn__D_e


  [WARN] mergem_iBsu1121VCorrected_trans_bigg: no boundary reaction for EX_fru_e
                condition  experimental growth  iBsu1103
0                 Glucose                 0.59     0.621
1               Gluconate                 0.42     0.000
2                Glycerol                 0.40     0.563
3                  Malate                 0.57     0.621
4         Malate; Glucose                 0.75     0.621
5                Pyruvate                 0.17     0.539
6  Succinate; L-Glutamate                 0.22     0.526
7                Fructose                 0.53     0.000 



  [WARN] mergem_iBsu1103v2_trans_bigg: no boundary reaction for EX_glcn__D_e


  [WARN] mergem_iBsu1103v2_trans_bigg: no boundary reaction for EX_fru_e
                condition  experimental growth  iBsu1103v2
0                 Glucose                 0.59       0.625
1               Gluconate                 0.42       0.000
2                Glycerol                 0.40       0.000
3                  Malate                 0.57       0.000
4         Malate; Glucose                 0.75       0.625
5                Pyruvate                 0.17       0.000
6  Succinate; L-Glutamate                 0.22       0.192
7                Fructose                 0.53       0.000 



Growth now tracks the substrate (drops for the poorer carbon sources, rises when malate is
combined with glucose) instead of being pinned at ~407/227 1/h everywhere. `Gluconate` and
`Fructose` still fail for both models and `Malate`/`Glycerol`/`Pyruvate` fail for `iBsu1103v2` -
the `medium_df["ID"]` to `EX_<id>_b` renaming scheme used here doesn't match those metabolites'
actual `_b` reaction IDs (e.g. gluconate/fructose use a different suffix), and in `iBsu1103v2`
some of those carbon sources hit genuine biosynthetic gaps once the free ride through the `_b`
layer is closed. Chasing each one down individually is the same kind of per-metabolite tracing
done above for glucose/water/ammonium - straightforward but tedious, and left out here since the
core diagnosis (the hidden `_b` layer) is already demonstrated.

## `iBsu1209`: the *initial* (pre-BiGG-conversion) model isn't clean either

The hypothesis going in was that `00_initial/etiBsu1209/iBsu1209.xml` - the ModelSEED-style draft,
before the `mergem` BiGG-namespace translation that left ~52% of `02_final_annotated/iBsu1209.xml`'s
metabolites as disconnected `_c[c]`-suffixed orphans (see the earlier investigation in this
conversation) - would be free of that particular problem and thus able to grow normally. It
doesn't have *that* bug, but it turns out to have its own, independent set of problems that also
block growth entirely (0 1/h under every condition, same as the converted version).

In [7]:
initial_path = os.path.join("..", "data", "models", "00_initial", "etiBsu1209", "iBsu1209.xml")
model = read_sbml_model(initial_path)

# M9-equivalent exchanges, found by tracing each M9 metabolite (by BiGG-style _e id) to its
# native ModelSEED "E#####" exchange reaction id - this model doesn't use BiGG reaction ids at all.
m9_native = {
    "E00001": (-1000, 1000),  # h2o
    "E00002": (-18, 0),       # o2
    "E00003": (-5, 1000),     # pi
    "E00004": (-1000, 1000),  # co2
    "E00006": (-5, 1000),     # nh4
    "E00023": (-5, 1000),     # so4
    "E00030": (-1000, 1000),  # ca2
    "E00033": (-1000, 1000),  # h
    "E00083": (-1000, 1000),  # k
    "E00095": (-8.7, 1000),   # D-glucose
    "E00101": (-1000, 1000),  # mg2
    "E00150": (-1000, 1000),  # na1
    "E00183": (-1000, 1000),  # fe2
    "E00184": (-1000, 1000),  # fe3
}
for r in model.exchanges:
    if r.lower_bound < 0:
        r.bounds = (0, r.upper_bound)
for rid, bounds in m9_native.items():
    model.reactions.get_by_id(rid).bounds = bounds

print(f"growth, M9 media, before any patch: {model.slim_optimize():.4f} 1/h")

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


growth, M9 media, before any patch: 0.0000 1/h


Zero, same as before. Tracing every biomass precursor with a one-way `demand` reaction (mirrors
the approach used above for `iBsu1103`) shows the whole biomass equation is blocked - even trivial
things like `h2o_c[c]` or `k_c[c]`, which should be freely producible from the open exchanges.
The actual culprit is the protein-synthesis pseudo-reaction `rxn05296`, which lumps all 20 charged
tRNAs into one reaction producing the aggregate `pro_c[c]` ("Protein") biomass precursor:

```
0.3653 argtrna_c[c] + ... + 0.5807 valtrna_c[c] <=>
    pro_c[c] + 0.5051 trnaala_c[c] + ... + 0.3653 trnal_L_c[c] + ... + 0.5807 trnaval_c[c]
```

Two things are wrong with it, both present already in this "clean" initial model (not introduced
by the later BiGG conversion):

1. **A curation typo.** The reactant list has 19 charged tRNAs, not 20 - **alanine's charged
   tRNA is missing**. Instead, the product side has a stray `trnal_L_c[c]` with no counterpart
   anywhere else in the model (a dead metabolite, likely meant to be `alatrna_c[c]` and mistyped).
2. **A real, separate gap.** Alanine has no aminoacyl-tRNA synthetase reaction in this
   reconstruction at all (no `alatrna_c[c]` metabolite exists anywhere), so even the *correctly*
   named product, `trnaala_c[c]` (uncharged tRNA-Ala, coefficient `0.5051`), has no consuming
   reaction - a genuine dead end, not a naming bug.

The first instinct - add a bidirectional `sink` reaction so `trnal_L_c[c]` stops mass-balance-locking
the pseudo-reaction to zero - works numerically (growth jumps to ~0.62 1/h) but is **not valid**:
a `sink` allows free import as well as export, and testing confirms the model then "grows" at the
same ~0.62 1/h with *every* exchange closed, including glucose - i.e. it's manufacturing biomass
from nothing through that one reaction, not really running metabolism.

In [8]:
# minimal, mass-conservative patch: drop the erroneous trnal_L_c[c] term (curation typo, #1)
rxn05296 = model.reactions.get_by_id("rxn05296")
stray = model.metabolites.get_by_id("trnal_L_c[c]")
rxn05296.subtract_metabolites({stray: rxn05296.get_coefficient(stray.id)})

# trnaala_c[c] is now a genuine dead end (#2, no charging reaction exists for Ala) - a one-way
# `demand` (efflux only, met --> ) lets it leave without allowing free import, unlike `sink`
model.add_boundary(model.metabolites.get_by_id("trnaala_c[c]"), type="demand")

print(f"growth, M9 media, glucose open:  {model.slim_optimize():.4f} 1/h")
model.reactions.get_by_id("E00095").lower_bound = 0.0
print(f"growth, M9 media, glucose CLOSED: {model.slim_optimize():.4f} 1/h  (sanity check - should be 0)")

growth, M9 media, glucose open:  0.6273 1/h
growth, M9 media, glucose CLOSED: 0.6273 1/h  (sanity check - should be 0)


Both patches are individually correct (typo fix + mass-conservative dead-end drain), but the
sanity check fails: growth is identical (~0.627 1/h) whether or not glucose - or any carbon
source at all - is available. `pfba()` on this solution shows `EX_glc__D` (`E00095`) running in
**reverse** (the model *excretes* glucose it never imported, i.e. it's running gluconeogenesis
from nothing) while heavy, unconstrained H+/O2 exchange sits in the background. Running
`cobra.flux_analysis.loopless_solution` on it returns the *same* objective value, which means
this isn't a simple futile cycle either (loopless FBA would have zeroed a pure futile cycle) - it's
a genuine, if unrealistic, steady-state pathway from `H2O`/`CO2`/inorganic salts to biomass that
the model's network topology permits. That points to a third, structural bug (most likely a
wrongly-reversible transporter or redox reaction somewhere in central metabolism or the electron
transport chain, in the same family as `iBsu1103`'s `R_H2Otg` issue) rather than anything in the
biomass equation - but with ~1950 reactions and no candidate short-list, isolating exactly which
reaction(s) enable it needs a systematic flux-variability sweep, which wasn't run here for time
reasons.

**Bottom line:** the initial model is not free of problems - it has at least three, two of which
are fixed above (curation typo; missing Ala-tRNA-synthetase gap patched conservatively), and a
third, more serious one (an apparent zero-substrate growth loop) left unresolved. Unlike
`iBsu1103`/`iBsu1103v2` above, the ~0.6 1/h figure for `iBsu1209` should **not** be read as a
validated, substrate-dependent growth rate yet.